# Week 2 — Second-Order ODEs & Linear Systems

> **Differential Equations for Scientists & Engineers**  
> *Characteristic equations, phase planes, and the geometry of coupled oscillators.*

---

## Learning Objectives

1. Solve **homogeneous linear second-order ODEs** using the characteristic equation
2. Apply **variation of parameters** and **undetermined coefficients** for forced equations
3. Convert higher-order ODEs to **first-order systems** $\mathbf{x}' = A\mathbf{x}$
4. Classify equilibria using **eigenvalue analysis** (node, saddle, spiral, center)
5. Draw **phase portraits** and **nullclines** from scratch
6. Analyse the **damped harmonic oscillator** in all damping regimes


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.patches import FancyArrowPatch

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'serif',
})

---

## 1. Homogeneous Linear Second-Order ODEs

The general form is:

$$a\,y'' + b\,y' + c\,y = 0$$

Try $y = e^{\lambda t}$. Substituting gives the **characteristic equation**:

$$a\lambda^2 + b\lambda + c = 0 \quad \Rightarrow \quad \lambda = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}$$

| Discriminant $\Delta = b^2 - 4ac$ | Roots | General Solution |
|---|---|---|
| $\Delta > 0$ | $\lambda_1 \neq \lambda_2 \in \mathbb{R}$ | $y = C_1 e^{\lambda_1 t} + C_2 e^{\lambda_2 t}$ |
| $\Delta = 0$ | $\lambda_1 = \lambda_2 = \lambda$ | $y = (C_1 + C_2 t)\,e^{\lambda t}$ |
| $\Delta < 0$ | $\lambda = \alpha \pm i\beta$ | $y = e^{\alpha t}(C_1\cos\beta t + C_2\sin\beta t)$ |

### 1.1 The Damped Harmonic Oscillator

$$m\,x'' + c\,x' + k\,x = 0$$

Dividing by $m$: $x'' + 2\gamma\,x' + \omega_0^2\,x = 0$ where $\gamma = c/(2m)$, $\omega_0 = \sqrt{k/m}$.

In [ ]:
def damped_oscillator(t, x0, v0, gamma, omega0):
    """
    Analytical solution to x'' + 2*gamma*x' + omega0^2*x = 0.
    Handles all three damping regimes.
    """
    Delta = gamma**2 - omega0**2

    if Delta > 1e-10:   # Overdamped
        r1 = -gamma + np.sqrt(Delta)
        r2 = -gamma - np.sqrt(Delta)
        # Solve for C1, C2 from ICs: x(0)=x0, x'(0)=v0
        C2 = (v0 - r1 * x0) / (r2 - r1)
        C1 = x0 - C2
        return C1 * np.exp(r1 * t) + C2 * np.exp(r2 * t), 'Overdamped'

    elif Delta < -1e-10:  # Underdamped
        omega_d = np.sqrt(-Delta)
        C1 = x0
        C2 = (v0 + gamma * x0) / omega_d
        return (np.exp(-gamma * t) *
                (C1 * np.cos(omega_d * t) + C2 * np.sin(omega_d * t))), 'Underdamped'

    else:                  # Critically damped
        C1 = x0
        C2 = v0 + gamma * x0
        return (C1 + C2 * t) * np.exp(-gamma * t), 'Critically damped'


t = np.linspace(0, 10, 1000)
omega0 = 2.0
x0, v0 = 1.0, 0.0

configs = [
    (0.3, 'Underdamped', '#2196F3'),
    (omega0, 'Critically damped', '#FF9800'),
    (3.5, 'Overdamped', '#F44336'),
]

fig, ax = plt.subplots(figsize=(10, 5))
for gamma, label, color in configs:
    y, regime = damped_oscillator(t, x0, v0, gamma, omega0)
    ax.plot(t, y, color=color, lw=2, label=f'{regime} ($\\gamma={gamma}$)')

ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('Time t'); ax.set_ylabel('Displacement x(t)')
ax.set_title(f'Damped Harmonic Oscillator ($\\omega_0={omega0}$, $x_0={x0}$, $v_0={v0}$)')
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

---

## 2. Wronskian & Linear Independence

Given two solutions $y_1, y_2$ of a homogeneous linear 2nd-order ODE, their **Wronskian** is:

$$W(y_1,y_2)(x) = \begin{vmatrix} y_1 & y_2 \\ y_1' & y_2' \end{vmatrix} = y_1 y_2' - y_2 y_1'$$

**Abel's Theorem:** If $y_1, y_2$ are solutions of $y'' + p(x)y' + q(x)y = 0$, then:
$$W(x) = W(x_0)\,\exp\!\left(-\int_{x_0}^{x} p(t)\,dt\right)$$
So $W$ is **either always zero or never zero** on an interval where $p,q$ are continuous.

**Key result:** $y_1, y_2$ form a **fundamental set of solutions** (i.e., are linearly independent) if and only if $W(y_1,y_2)(x) \neq 0$.

**Example:** For $y'' - 3y' + 2y = 0$, solutions $y_1 = e^x$, $y_2 = e^{2x}$:
$$W = e^x \cdot 2e^{2x} - e^{2x} \cdot e^x = e^{3x} \neq 0$$

In [ ]:
def wronskian(y1, y2, dy1, dy2):
    """Compute W(y1,y2) = y1*y2' - y2*y1' at each point in an array."""
    return y1 * dy2 - y2 * dy1


def numerical_wronskian(sol1_func, sol2_func, x_arr, h=1e-7):
    """
    Estimate Wronskian numerically via central differences.
    sol1_func, sol2_func: callable y(x)
    """
    y1  = sol1_func(x_arr)
    y2  = sol2_func(x_arr)
    dy1 = (sol1_func(x_arr+h) - sol1_func(x_arr-h)) / (2*h)
    dy2 = (sol2_func(x_arr+h) - sol2_func(x_arr-h)) / (2*h)
    return wronskian(y1, y2, dy1, dy2)


x = np.linspace(0.1, 3, 400)

# ── Case A: Linearly INDEPENDENT — y1=e^x, y2=e^{2x} (W = e^{3x} > 0) ──
W_A = numerical_wronskian(np.exp, lambda t: np.exp(2*t), x)
W_A_exact = np.exp(3*x)

# ── Case B: Linearly DEPENDENT — y1=sin(x), y2=2sin(x) (W = 0) ──────────
W_B = numerical_wronskian(np.sin, lambda t: 2*np.sin(t), x)

# ── Abel's theorem verification for y''+y=0 (p=0, W=const) ──────────────
# Solutions: cos(x), sin(x) — W = 1 everywhere
W_cos_sin = numerical_wronskian(np.cos, np.sin, x)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(x, W_A, lw=2.5, color='#27ae60', label='Numerical $W$')
axes[0].plot(x, W_A_exact, lw=1.5, ls='--', color='#2c3e50', label=r'Exact $e^{3x}$')
axes[0].set_title(r'Independent: $y_1=e^x,\;y_2=e^{2x}$', fontsize=11)
axes[0].set_xlabel('x'); axes[0].set_ylabel('W')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(x, W_B, lw=2.5, color='#e74c3c', label='W = 0 (dependent)')
axes[1].set_title(r'Dependent: $y_1=\sin x,\;y_2=2\sin x$', fontsize=11)
axes[1].set_xlabel('x'); axes[1].set_ylabel('W')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(x, W_cos_sin, lw=2.5, color='#8e44ad',
             label="Abel: $W=1$ (p=0)")
axes[2].set_title(r"Abel's thm: $y''+y=0$, $W=\mathrm{const}$", fontsize=11)
axes[2].set_xlabel('x'); axes[2].set_ylabel('W')
axes[2].legend(); axes[2].grid(alpha=0.3)

fig.suptitle('Wronskian & Linear Independence', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Case A — max error vs exact Wronskian: {np.max(np.abs(W_A - W_A_exact)):.2e}')
print(f'Case B — max |W| (should be ~0):       {np.max(np.abs(W_B)):.2e}')
print(f'Abel   — W range (should be ~1):       [{W_cos_sin.min():.6f}, {W_cos_sin.max():.6f}]')

---

## 3. Variation of Parameters

For the **nonhomogeneous** equation $y'' + p(t)y' + q(t)y = g(t)$, given a fundamental pair $\{y_1, y_2\}$ solving the homogeneous equation, the particular solution is:

$$y_p = -y_1\int\frac{y_2\,g}{W}\,dt + y_2\int\frac{y_1\,g}{W}\,dt$$

where $W = y_1 y_2' - y_2 y_1'$ is the **Wronskian**.

In [ ]:
def variation_of_parameters(y1, y2, dy1, dy2, g_func, t_arr):
    """
    Compute particular solution via variation of parameters using cumulative trapezoidal.
    y1, y2: fundamental solutions (arrays)
    dy1, dy2: their derivatives (arrays)
    g_func: forcing function callable
    """
    dt = t_arr[1] - t_arr[0]
    g = np.array([g_func(ti) for ti in t_arr])
    W = y1 * dy2 - y2 * dy1                 # Wronskian

    u1 = np.cumsum(-y2 * g / W) * dt        # u1' = -y2*g/W
    u2 = np.cumsum(y1 * g / W) * dt         # u2' = y1*g/W

    return u1 * y1 + u2 * y2


# --- Example: y'' + y = sec(t)  on  t in [0.1, pi/2 - 0.1] ---
# Fundamental: y1 = cos(t), y2 = sin(t)
# Particular (known): y_p = cos(t)*ln|cos(t)| + t*sin(t)

t = np.linspace(0.1, np.pi/2 - 0.1, 400)
y1  = np.cos(t);   y2  = np.sin(t)
dy1 = -np.sin(t);  dy2 = np.cos(t)
g_func = lambda t: 1.0 / np.cos(t)

y_p_num = variation_of_parameters(y1, y2, dy1, dy2, g_func, t)
y_p_exact = np.cos(t) * np.log(np.abs(np.cos(t))) + t * np.sin(t)

# Adjust for the integration constant
y_p_num -= y_p_num[0] - y_p_exact[0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, y_p_exact, 'k-', lw=2, label='Exact $y_p$')
ax.plot(t, y_p_num, 'r--', lw=1.5, label='Variation of Parameters (numerical)')
ax.set_xlabel('t'); ax.set_ylabel('$y_p(t)$')
ax.set_title("Particular Solution: $y'' + y = \\sec(t)$")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

---

## 4. Power Series Solutions

When an ODE has a **ordinary point** at $x_0$, we assume a solution of the form:
$$y = \sum_{n=0}^{\infty} a_n (x-x_0)^n$$

Substituting and matching powers of $(x-x_0)^n$ yields a **recurrence relation** for $a_n$.

**Example — Airy equation:** $y'' - xy = 0$

Substituting $y = \sum a_n x^n$ gives:
$$\sum_{n=2}^{\infty} n(n-1)a_n x^{n-2} - \sum_{n=0}^{\infty} a_n x^{n+1} = 0$$
Shifting indices and matching: $a_{n+3} = \dfrac{a_n}{(n+3)(n+2)}$, $\;a_2 = 0$.

This produces **two linearly independent series** (determined by $a_0$ and $a_1$) — the Airy functions $\mathrm{Ai}(x)$ and $\mathrm{Bi}(x)$.

**Frobenius method:** At a **regular singular point** $x_0$, try:
$$y = x^r \sum_{n=0}^{\infty} a_n x^n$$
The **indicial equation** for $r$ is obtained from the lowest-power term.

In [ ]:
import math
def power_series_airy(x_arr, a0=1.0, a1=0.0, N_terms=40):
    """
    Power series solution of the Airy equation y'' - x*y = 0.
    Recurrence: a_{n+3} = a_n / ((n+3)(n+2)),  a_2 = 0.

    Two independent solutions:
      y1: a0=1, a1=0  (even-ish series)
      y2: a0=0, a1=1  (odd-ish series)
    General solution: y = a0*y1 + a1*y2.
    """
    # build coefficient array
    a = np.zeros(N_terms)
    a[0], a[1] = a0, a1
    # a[2] = 0 always
    for n in range(N_terms - 3):
        a[n+3] = a[n] / ((n+3)*(n+2))

    # evaluate series at each x
    result = np.zeros_like(x_arr, dtype=float)
    for n in range(N_terms):
        result += a[n] * x_arr**n
    return result


def frobenius_bessel_j0(x_arr, N_terms=30):
    """
    Frobenius / power-series computation of J_0(x) (Bessel function of order 0).
    Bessel's equation: x^2 y'' + x y' + x^2 y = 0.
    Indicial equation: r^2 = 0  →  r = 0.
    Recurrence (r=0): a_n = -a_{n-2} / (n^2),  n >= 2,  a_1 = 0.
    J_0(x) = sum_{m=0}^inf  (-1)^m / (m!)^2  * (x/2)^{2m}
    """
    result = np.zeros_like(x_arr, dtype=float)
    for m in range(N_terms):
        coeff = ((-1)**m) / (float(math.factorial(m))**2)
        result += coeff * (x_arr/2)**(2*m)
    return result


from scipy.special import airy as scipy_airy, j0 as scipy_j0

x_pos = np.linspace(0, 4, 300)        # Airy series converges for all x
x_neg = np.linspace(-4, 4, 300)       # show oscillatory Ai on negative side
x_bes = np.linspace(0.01, 10, 400)

# Airy Ai: normalised combination a0=Ai(0), a1=Ai'(0)
from scipy.special import airy as _airy
Ai0, Aip0, _, _ = _airy(0.0)
y_series_ai = power_series_airy(x_neg, a0=Ai0, a1=Aip0, N_terms=50)
Ai_exact, _, _, _ = scipy_airy(x_neg)

# Bessel J0
y_series_j0 = frobenius_bessel_j0(x_bes, N_terms=25)
y_exact_j0  = scipy_j0(x_bes)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(x_neg, y_series_ai, lw=2.5, color='#e74c3c', label='Power series (N=50)')
ax.plot(x_neg, Ai_exact,    lw=1.5, ls='--', color='#2c3e50', label='scipy Ai(x)')
ax.set_title("Airy equation: $y'' - xy = 0$", fontsize=12)
ax.set_ylim(-0.8, 0.8); ax.set_xlabel('x'); ax.set_ylabel('Ai(x)')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(x_bes, y_series_j0, lw=2.5, color='#3498db', label='Frobenius series (N=25)')
ax.plot(x_bes, y_exact_j0,  lw=1.5, ls='--', color='#2c3e50', label='scipy $J_0(x)$')
ax.set_title(r"Bessel eq. (order 0): Frobenius gives $J_0(x)$", fontsize=12)
ax.set_xlabel('x'); ax.set_ylabel('$J_0(x)$')
ax.legend(); ax.grid(alpha=0.3)

fig.suptitle('Power Series & Frobenius Method', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Airy  max error (|x|<=4): {np.max(np.abs(y_series_ai - Ai_exact)):.2e}')
print(f'J_0   max error (x<=10):  {np.max(np.abs(y_series_j0 - y_exact_j0)):.2e}')

---

## 5. Systems of ODEs — Matrix Eigenvalue Method

A first-order linear system $\mathbf{x}' = A\mathbf{x}$ (constant $A$) is solved via eigendecomposition.

**Algorithm:**
1. Compute eigenvalues $\lambda_i$ and eigenvectors $\mathbf{v}_i$ of $A$
2. Each pair contributes a fundamental solution $e^{\lambda_i t}\mathbf{v}_i$
3. General solution: $\mathbf{x}(t) = \sum_i c_i e^{\lambda_i t}\mathbf{v}_i$
4. Apply initial conditions to find $c_i$

**Cases:**

| Eigenvalues | Behavior | Phase portrait |
|-------------|----------|---------------|
| Real, distinct | Exponential growth/decay | Node |
| Real, repeated | Algebraic + exponential | Degenerate node |
| Complex $\alpha \pm \beta i$ | Oscillatory + exp envelope | Spiral / center |
| Pure imaginary | Periodic | Elliptic center |

**Fundamental matrix:** $\Phi(t) = [\mathbf{x}_1(t)\;|\;\mathbf{x}_2(t)]$ satisfies $\Phi' = A\Phi$, $\;\Phi(0)=I$.

In [ ]:
def matrix_ode_solution(A, x0, t_arr):
    """
    Solve x' = A x,  x(0) = x0  via eigendecomposition.
    Works for real distinct and complex eigenvalues.
    Returns real part (imaginary part is ~0 for real ICs).
    """
    eigenvalues, V = np.linalg.eig(A)   # V[:,i] is eigenvector for eigenvalues[i]
    # coefficients: c = V^{-1} x0
    c = np.linalg.solve(V, x0.astype(complex))
    n = len(x0)
    x = np.zeros((len(t_arr), n), dtype=complex)
    for i in range(n):
        x += c[i] * np.exp(eigenvalues[i] * t_arr[:, None]) * V[:, i]
    return x.real


def fundamental_matrix(A, t_arr, h=None):
    """
    Build the fundamental matrix Phi(t) via matrix exponential using
    eigendecomposition: Phi(t) = V diag(exp(lambda_i t)) V^{-1}.
    """
    eigenvalues, V = np.linalg.eig(A)
    V_inv = np.linalg.inv(V)
    Phi = np.zeros((len(t_arr), A.shape[0], A.shape[1]), dtype=complex)
    for k, t in enumerate(t_arr):
        D = np.diag(np.exp(eigenvalues * t))
        Phi[k] = V @ D @ V_inv
    return Phi.real


t = np.linspace(0, 4*np.pi, 800)
x0 = np.array([2.0, 0.0])

# ── Four canonical systems ─────────────────────────────────────────────────
systems = [
    {'A': np.array([[-1., 2.],[0., -3.]]),  'label': 'Stable node\n$\\lambda=-1,-3$'},
    {'A': np.array([[0., -2.],[2.,  0.]]),  'label': 'Centre (pure imag.)\n$\\lambda=\\pm 2i$'},
    {'A': np.array([[-0.2,-2.],[2., -0.2]]),'label': 'Stable spiral\n$\\lambda=-0.2\\pm2i$'},
    {'A': np.array([[1., 1.],[0.,  2.]]),   'label': 'Unstable node\n$\\lambda=1,2$'},
]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for col, sys in enumerate(systems):
    A_sys = sys['A']
    lam = np.linalg.eigvals(A_sys)
    ax_traj  = axes[0, col]
    ax_phase = axes[1, col]

    # Time-domain trajectories from several ICs
    colors = ['#e74c3c','#3498db','#27ae60','#f39c12']
    ics = [np.array([2,0]), np.array([-2,0]), np.array([0,2]), np.array([0,-2])]
    for ic, col_c in zip(ics, colors):
        traj = matrix_ode_solution(A_sys, ic, t)
        ax_traj.plot(t, traj[:,0], color=col_c, lw=1.5, alpha=0.8)
    ax_traj.set_title(f'$x_1(t)$ — {sys["label"]}', fontsize=9)
    ax_traj.set_xlabel('t'); ax_traj.grid(alpha=0.3)

    # Phase portrait
    G = 3.0
    yy, xx = np.mgrid[-G:G:20j, -G:G:20j]
    UU = A_sys[0,0]*xx + A_sys[0,1]*yy
    VV = A_sys[1,0]*xx + A_sys[1,1]*yy
    spd = np.sqrt(UU**2 + VV**2) + 1e-9
    ax_phase.quiver(xx, yy, UU/spd, VV/spd, spd, cmap='cool', alpha=0.6)
    for ic, col_c in zip(ics, colors):
        traj = matrix_ode_solution(A_sys, ic, t)
        ax_phase.plot(traj[:,0], traj[:,1], color=col_c, lw=1.5, alpha=0.9)
    ax_phase.set_xlim(-G,G); ax_phase.set_ylim(-G,G)
    ax_phase.set_xlabel('$x_1$'); ax_phase.set_ylabel('$x_2$')
    lam_str = ', '.join(f'{v:.2f}' for v in lam)
    ax_phase.set_title(f'Phase portrait  $\\lambda$=[{lam_str}]', fontsize=9)
    ax_phase.grid(alpha=0.3)

fig.suptitle("Systems $\\mathbf{x}' = A\\mathbf{x}$ — Eigenvalue Method", fontsize=14)
plt.tight_layout()
plt.show()

# Verification: fundamental matrix at t=0 should be identity
Phi = fundamental_matrix(systems[2]['A'], np.array([0.0]))
print('Phi(0) (should be I):\n', np.round(Phi[0], 6))

---

## 6. Linear Systems and Phase Portraits

Any $n$th-order ODE can be written as the first-order system $\mathbf{x}' = A\mathbf{x}$.

For the 2D system, the **eigenvalues** $\lambda_1, \lambda_2$ of $A$ determine the topology:

| Eigenvalue type | Equilibrium type | Stable? |
|---|---|---|
| Both real, same sign | Node (stable/unstable) | $\text{Re}(\lambda) < 0$ |
| Both real, opposite signs | Saddle | Never |
| Complex conjugates $\alpha \pm i\beta$ | Spiral | $\alpha < 0$ |
| Pure imaginary $\pm i\beta$ | Center | Neutrally stable |


In [ ]:
def phase_portrait(A, x_range=(-3, 3), y_range=(-3, 3), n=20, ax=None, title=''):
    """Draw phase portrait for x' = Ax by computing the vector field on a grid."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))

    x_vals = np.linspace(*x_range, n)
    y_vals = np.linspace(*y_range, n)
    X, Y = np.meshgrid(x_vals, y_vals)
    UV = A @ np.array([X.ravel(), Y.ravel()])
    U, V = UV[0].reshape(n, n), UV[1].reshape(n, n)

    norm = np.sqrt(U**2 + V**2) + 1e-12
    ax.quiver(X, Y, U/norm, V/norm, alpha=0.5, color='steelblue',
              width=0.003, headwidth=3)

    # Overlay some trajectories via forward Euler
    np.random.seed(42)
    r = np.random.uniform(1.5, 2.5, 12)
    theta = np.linspace(0, 2*np.pi, 12, endpoint=False)
    ics = np.column_stack([r * np.cos(theta), r * np.sin(theta)])

    colors = cm.plasma(np.linspace(0.1, 0.9, len(ics)))
    dt = 0.02
    N_steps = 400

    for ic, c in zip(ics, colors):
        traj = [ic.copy()]
        state = ic.copy()
        for _ in range(N_steps):
            state = state + dt * (A @ state)
            if np.linalg.norm(state) > 6:
                break
            traj.append(state.copy())
        traj = np.array(traj)
        ax.plot(traj[:, 0], traj[:, 1], color=c, lw=1.2, alpha=0.85)

    eigs = np.linalg.eigvals(A)
    ax.set_xlim(*x_range); ax.set_ylim(*y_range)
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.set_title(f'{title}\n$\\lambda = {eigs[0]:.2f}, {eigs[1]:.2f}$')
    return ax


# Four canonical equilibrium types
cases = [
    (np.array([[-1, 0], [0, -2]]), 'Stable Node'),
    (np.array([[1, 0], [0, -2]]), 'Saddle'),
    (np.array([[-0.3, -2], [2, -0.3]]), 'Stable Spiral'),
    (np.array([[0, -2], [2, 0]]), 'Center'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for ax, (A, title) in zip(axes.ravel(), cases):
    phase_portrait(A, ax=ax, title=title)

plt.tight_layout(); plt.show()

---

## 7. Resonance in Forced Oscillators

The forced oscillator:

$$x'' + 2\gamma\,x' + \omega_0^2\,x = F_0\cos(\omega t)$$

has a particular solution $x_p = A(\omega)\cos(\omega t - \phi)$ where the **amplitude response** is:

$$A(\omega) = \frac{F_0}{\sqrt{(\omega_0^2 - \omega^2)^2 + 4\gamma^2\omega^2}}$$

**Resonance** occurs when $\omega = \omega_r = \sqrt{\omega_0^2 - 2\gamma^2}$.

In [ ]:
omega_drive = np.linspace(0.1, 4, 500)
omega0 = 2.0
F0 = 1.0
gammas = [0.1, 0.3, 0.7, 1.2]

fig, ax = plt.subplots(figsize=(9, 5))
colors = cm.inferno(np.linspace(0.15, 0.85, len(gammas)))

for gamma, c in zip(gammas, colors):
    A_omega = F0 / np.sqrt((omega0**2 - omega_drive**2)**2 + 4*gamma**2*omega_drive**2)
    omega_r = np.sqrt(max(omega0**2 - 2*gamma**2, 0))
    ax.plot(omega_drive, A_omega, color=c, lw=2, label=f'$\\gamma={gamma}$')
    if omega_r > 0:
        ax.axvline(omega_r, color=c, ls=':', lw=1, alpha=0.6)

ax.axvline(omega0, color='k', ls='--', lw=1, label=f'$\\omega_0={omega0}$')
ax.set_xlabel('Driving frequency $\\omega$')
ax.set_ylabel('Amplitude $A(\\omega)$')
ax.set_title('Resonance Curves — Forced Damped Oscillator')
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

---

## 8. Exercises

1. **(Characteristic equation)** Solve $y'' - 5y' + 6y = 0$ with $y(0) = 1$, $y'(0) = 0$. Verify numerically with forward Euler.

2. **(Repeated roots)** Solve $y'' - 4y' + 4y = 0$ and explain why two linearly independent solutions require the factor $t$.

3. **(VoP)** Solve $y'' + 4y = \tan(2t)$ using variation of parameters. Plot $y_h + y_p$.

4. **(Phase portrait)** Classify the equilibrium of $A = \begin{pmatrix}1 & -4\\1 & -1\end{pmatrix}$ and draw the phase portrait.

5. **(Nonlinear preview)** The pendulum satisfies $\theta'' + \sin\theta = 0$. Linearise around $\theta^* = 0$ and $\theta^* = \pi$, classify each equilibrium, and compare the linear phase portrait with the true nonlinear one.
6. **(Wronskian)** For $y'' + 4y = 0$ with solutions $\cos 2x$ and $\sin 2x$, compute $W$ analytically and verify Abel's theorem (note $p(x)=0$).
7. **(Power series)** Derive the first 6 non-zero terms of the power series solution of $y'' + xy' + y = 0$ about $x_0=0$ and plot against a numerical RK4 solution.
8. **(Matrix system)** Solve $\mathbf{x}' = \begin{pmatrix}2&1\\0&3\end{pmatrix}\mathbf{x}$ with $\mathbf{x}(0)=(1,1)^T$ using `matrix_ode_solution`. Verify $\mathbf{x}(1)$ analytically.
